### Step 1: Configuration & Imports
### We add scikit-learn for scaling and define which columns need to be normalized.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import drive
import os
import polars as pl
import shutil
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Updated Paths (Matching your new Medallion structure)
base_path = "/content/drive/My Drive/sec_data"

# These now point to your professional folder names
silver_dir = os.path.join(base_path, "02_silver_cleaned")
gold_dir   = os.path.join(base_path, "03_gold_features")
master_dir = os.path.join(base_path, "04_master_ml")

# 3. Create Bronze folder if you haven't yet (for your raw .tsv files)
bronze_dir = os.path.join(base_path, "01_bronze_raw")
os.makedirs(bronze_dir, exist_ok=True)

important_tags = [
    "Assets", "Liabilities", "StockholdersEquity", "Revenues",
    "NetIncomeLoss", "OperatingIncomeLoss", "AssetsCurrent",
    "LiabilitiesCurrent", "Inventory", "InterestExpense",
    "RetainedEarnings", "CashAndCashEquivalentsAtCarryingValue"
]

features_to_scale = [
    "Assets", "Revenues", "NetIncomeLoss",
    "current_ratio", "quick_ratio", "cash_ratio",
    "roa", "profit_margin", "operating_margin", "roe",
    "debt_to_assets", "debt_to_equity", "asset_turnover",
    "interest_coverage", "retained_earnings_ratio"
]

print(f"✅ Paths updated! Ready to process data into {master_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Paths updated! Ready to process data into /content/drive/My Drive/sec_data/04_master_ml


### Step 2: The Hybrid Engine & Execution Loop
### This is your existing core logic, which handles the 2024 (Quarterly) and 2025 (Monthly) data in Parquet format.
### Function Name Alignment: Changed process_quarter_to_parquet_logic to process_quarter_to_df in both the definition and the loop.

### Return vs Write: The function now returns a polars.DataFrame instead of writing a file, allowing the loop to collect and merge all quarters for a specific year before saving.

### Automatic Year Detection: The loop now groups your 2022, 2023, 2024, and 2025 folders correctly to ensure each year gets its own "security" Parquet file.

In [9]:
def process_single_folder(folder_path, tags):
    num_path = os.path.join(folder_path, 'num.tsv')
    sub_path = os.path.join(folder_path, 'sub.tsv')
    if not os.path.exists(num_path) or not os.path.exists(sub_path): return None

    try:
        num_lazy = pl.scan_csv(num_path, separator='\t', ignore_errors=True) \
            .filter(pl.col("tag").is_in(tags)) \
            .filter(pl.col("qtrs").is_in([0, 1, 4])) \
            .select([pl.col("adsh"), pl.col("tag"), pl.col("value").cast(pl.Float32), pl.col("ddate")])

        df_pivoted = num_lazy.collect(streaming=True).pivot(
            values="value", index=["adsh", "ddate"], on="tag", aggregate_function="first"
        )

        for tag in tags:
            if tag not in df_pivoted.columns:
                df_pivoted = df_pivoted.with_columns(pl.lit(None).cast(pl.Float32).alias(tag))

        df_pivoted = df_pivoted.lazy().with_columns([
            (pl.col("AssetsCurrent") / pl.col("LiabilitiesCurrent")).alias("current_ratio"),
            ((pl.col("AssetsCurrent") - pl.col("Inventory")) / pl.col("LiabilitiesCurrent")).alias("quick_ratio"),
            (pl.col("CashAndCashEquivalentsAtCarryingValue") / pl.col("LiabilitiesCurrent")).alias("cash_ratio"),
            (pl.col("NetIncomeLoss") / pl.col("Assets")).alias("roa"),
            (pl.col("NetIncomeLoss") / pl.col("Revenues")).alias("profit_margin"),
            (pl.col("OperatingIncomeLoss") / pl.col("Revenues")).alias("operating_margin"),
            (pl.col("NetIncomeLoss") / pl.col("StockholdersEquity")).alias("roe"),
            (pl.col("Liabilities") / pl.col("Assets")).alias("debt_to_assets"),
            (pl.col("Liabilities") / pl.col("StockholdersEquity")).alias("debt_to_equity"),
            (pl.col("Revenues") / pl.col("Assets")).alias("asset_turnover"),
            (pl.col("OperatingIncomeLoss") / pl.col("InterestExpense")).alias("interest_coverage"),
            (pl.col("RetainedEarnings") / pl.col("Assets")).alias("retained_earnings_ratio")
        ]).collect()

        sub_df = pl.read_csv(sub_path, separator='\t', ignore_errors=True) \
            .select([pl.col("adsh"), pl.col("cik").cast(pl.Int32), pl.col("name")])

        return df_pivoted.join(sub_df, on="adsh", how="left")
    except Exception as e:
        return None

# Find folders in Drive
folders = [f for f in os.listdir(base_path) if f.endswith('_notes')]
years_found = sorted(list(set([f[:4] for f in folders])))

for year in years_found:
    yearly_output_path = os.path.join(silver_dir, f"financials_{year}.parquet")
    if os.path.exists(yearly_output_path): continue
    print(f"🚀 Processing Year: {year}...")
    year_folders = [f for f in folders if f.startswith(year)]
    year_data_chunks = [process_single_folder(os.path.join(base_path, f), important_tags) for f in year_folders]
    year_data_chunks = [c for c in year_data_chunks if c is not None]
    if year_data_chunks:
        all_columns = year_data_chunks[0].columns
        standardized_chunks = [chunk.select(all_columns) for chunk in year_data_chunks]
        yearly_df = pl.concat(standardized_chunks)
        yearly_df.write_parquet(yearly_output_path, compression="zstd")

### Step 3: Handling Nulls & Normalization. LSTMs cannot handle NaN or Null values. Since you are building a time-series model, we will use Forward Fill (using the previous month's data if the current is missing) and then Standard Scaling.Standard scaling uses the formula:
### $$ z = \frac{x - \mu}{\sigma}$$

In [5]:
print("\n📦 Building Master Dataset & Scaling...")

# FIX: Changed 'output_dir' to 'gold_dir'
# Also changed "financials_*.parquet" to "*_gold.parquet" to match your file names
master_df = pl.read_parquet(os.path.join(gold_dir, "*_gold.parquet"))

master_df = master_df.sort(["cik", "ddate"]).unique(subset=["cik", "ddate"])

# --- Revenue Growth ---
master_df = master_df.with_columns(
    ((pl.col("Revenues") / pl.col("Revenues").shift(1).over("cik")) - 1).fill_null(0).alias("revenue_growth_rate")
)

# --- Persistence Flag ---
master_df = master_df.with_columns((pl.col("interest_coverage") < 1.5).cast(pl.Int32).alias("is_distressed"))
master_df = master_df.with_columns(
    pl.col("is_distressed").rolling_sum(window_size=2).over("cik").fill_null(0).alias("persistent_distress_flag")
)

new_features = ["revenue_growth_rate", "persistent_distress_flag"]
all_features_to_scale = features_to_scale + new_features

# Clean infinite values
master_df = master_df.with_columns([
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c) for c in all_features_to_scale
])

# Forward fill missing data
master_df = master_df.group_by("cik").map_groups(lambda group: group.fill_null(strategy="forward")).fill_null(0)

# --- Scaling ---
pdf = master_df.to_pandas()
scaler = StandardScaler()
pdf[all_features_to_scale] = scaler.fit_transform(pdf[all_features_to_scale].astype('float64'))
master_df = pl.from_pandas(pdf)

# Save the final model-ready file to '04_master_ml'
final_path = os.path.join(master_dir, "lstm_ready_data.parquet")
master_df.write_parquet(final_path)

print(f"✅ Data Scaled and saved in: {final_path}")


📦 Building Master Dataset & Scaling...
✅ Data Scaled and saved in: /content/drive/My Drive/sec_data/04_master_ml/lstm_ready_data.parquet


### Step 4: Labelling (The "Target" for the Crash)To predict a "Crash," you need a label ($y$). Usually, a crash is defined as a significant price drop (e.g., -15%) in the period following a filing. Note: You will need a CSV of stock prices (CIK, Date, Price) to merge here. I have provided the logic to create the "Target" column.

In [7]:
print("\n💾 Finalizing and Saving to Master ML Folder...")

# 1. Add target column
master_df = master_df.with_columns(pl.lit(0).alias("target_crash"))

# 2. Organize columns
metadata_cols = ["cik", "name", "adsh", "ddate"]
final_column_order = metadata_cols + all_features_to_scale + ["target_crash"]
master_df = master_df.select(final_column_order)

# 3. FIX: Changed 'final_output_dir' to 'master_dir'
# We are only saving the Parquet file to avoid "Format Bloat"
parquet_ml_path = os.path.join(master_dir, "lstm_ready_data.parquet")

# 4. Save the file
master_df.write_parquet(parquet_ml_path, compression="zstd")

print(f"✅ Final Dataset Saved to: {parquet_ml_path}")
print(f"🚀 Your data is now in the '04_master_ml' folder and ready for the LSTM!")


💾 Finalizing and Saving to Master ML Folder...
✅ Final Dataset Saved to: /content/drive/My Drive/sec_data/04_master_ml/lstm_ready_data.parquet
🚀 Your data is now in the '04_master_ml' folder and ready for the LSTM!


**Project Summary**

In [8]:
print("✨ Project Data Pipeline Complete!")
print("-" * 30)
print(f"📁 Silver Data (Cleaned): {silver_dir}")
print(f"📁 Gold Data (Features): {gold_dir}")
print(f"📁 Master ML File: {os.path.join(master_dir, 'lstm_ready_data.parquet')}")
print("-" * 30)
print("🚀 Everything is organized! You can now share the 'sec_data' link with your teammate.")

✨ Project Data Pipeline Complete!
------------------------------
📁 Silver Data (Cleaned): /content/drive/My Drive/sec_data/02_silver_cleaned
📁 Gold Data (Features): /content/drive/My Drive/sec_data/03_gold_features
📁 Master ML File: /content/drive/My Drive/sec_data/04_master_ml/lstm_ready_data.parquet
------------------------------
🚀 Everything is organized! You can now share the 'sec_data' link with your teammate.
